In [10]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(
    "../dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

# ==========================================
# CLEANING
# ==========================================

df["TotalCharges"] = df["TotalCharges"].replace(
    " ",
    np.nan
)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"]
)

df.dropna(inplace=True)

df.drop("customerID", axis=1, inplace=True)

# ==========================================
# BINARY ENCODING
# ==========================================

binary_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]

for col in binary_cols:

    df[col] = df[col].map({
        "Yes": 1,
        "No": 0,
        "Male": 1,
        "Female": 0
    })

# ==========================================
# ONE HOT ENCODING
# ==========================================

df = pd.get_dummies(
    df,
    drop_first=True
)

# ==========================================
# SPLIT
# ==========================================

X = df.drop("Churn", axis=1)

y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================================
# SCALING
# ==========================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# ==========================================
# MODELS
# ==========================================

models = {

    "Logistic Regression":
        LogisticRegression(max_iter=2000),

    "Decision Tree":
        DecisionTreeClassifier(),

    "Random Forest":
        RandomForestClassifier(),

    "KNN":
        KNeighborsClassifier(),

    "SVM":
        SVC(probability=True),

    "Naive Bayes":
        GaussianNB(),

    "XGBoost":
        XGBClassifier(
            eval_metric='logloss'
        ),

    "LightGBM":
        LGBMClassifier(),

    "CatBoost":
        CatBoostClassifier(
            verbose=0
        )
}

# ==========================================
# TRAINING
# ==========================================

results = []

for name, model in models.items():

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    y_prob = model.predict_proba(X_test_scaled)[:,1]

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    results.append([
        name,
        accuracy,
        roc_auc
    ])

    print("\n==============================")
    print(f"Model: {name}")
    print("==============================")

    print("Accuracy:", accuracy)

    print("ROC-AUC:", roc_auc)

    print("\nClassification Report:\n")

    print(
        classification_report(
            y_test,
            y_pred
        )
    )

# ==========================================
# FINAL RESULTS
# ==========================================

results_df = pd.DataFrame(

    results,

    columns=[
        "Model",
        "Accuracy",
        "ROC-AUC"
    ]
)

results_df = results_df.sort_values(
    by="ROC-AUC",
    ascending=False
)

print("\nFinal Results:\n")

print(results_df)


Model: Logistic Regression
Accuracy: 0.8038379530916845
ROC-AUC: 0.8356727976766699

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407


Model: Decision Tree
Accuracy: 0.7164179104477612
ROC-AUC: 0.6326234269119071

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.81      0.81      1033
           1       0.47      0.45      0.46       374

    accuracy                           0.72      1407
   macro avg       0.63      0.63      0.63      1407
weighted avg       0.71      0.72      0.72      1407


Model: Random Forest
Accuracy: 0.7867803837953091
ROC-AUC: 0.8179631000512498

Classification Report:

              precision    r